Test seti pathleri

In [1]:
import os
import json
import glob
import re

# --- Yapılandırma ---
# DUD-E'nin tüm hedef klasörlerinin bulunduğu ana dizin
# Sizin örneğinizde bu "/media/ubuntu/8TB/havvanur/PLANET_MODEL/data/datasets/all" olmalı
BASE_DIR = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/all" 

# Yeni JSON dosyasının adı ve yolu
OUTPUT_JSON = os.path.join(os.path.dirname(BASE_DIR), "DUD_E_structure_map_prepared.json")

# --- Ana Fonksiyon ---
def create_dude_structure_map(base_dir, output_json):
    """
    DUD-E klasörlerini tarar ve her aktif/decoy molekülü için bir yapısal harita oluşturur.
    DUD-E'de skorlama sadece tarama gücü (AUROC) üzerine kuruludur, bu yüzden pKd değeri 
    atanmaz (sadece is_binder = 1 veya 0 etiketi kullanılır).
    """
    if not os.path.isdir(base_dir):
        print(f"HATA: DUD-E ana dizini bulunamadı: {base_dir}")
        return {}

    structure_map = {}
    found_molecules = 0
    
    # 1. Adım: Tüm hedef protein klasörlerini tara
    # Örneğin: 'ada', 'abl1', 'adrb2', vb.
    target_folders = glob.glob(os.path.join(base_dir, "*/"))
    
    print(f"DUD-E hedefleri taranıyor ({len(target_folders)} adet)...")
    
    for folder_path in target_folders:
        target_name = os.path.basename(os.path.normpath(folder_path))
        
        # Ana protein PDB dosyasının yolu
        protein_pdb = os.path.join(folder_path, "receptor.pdb")
        
        # Protein PDB dosyasının varlığını kontrol et
        if not os.path.exists(protein_pdb):
            print(f"UYARI: {target_name} için receptor.pdb bulunamadı. Atlanıyor.")
            continue
            
        # 2. Adım: Aktif ve Decoy dosyalarını tara (genellikle .mol2.gz veya .sdf.gz)
        # Genellikle .mol2.gz en güvenilir formattır.
        
        # A) Aktifler (Binder'lar)
        active_files = glob.glob(os.path.join(folder_path, "actives_final.sdf.gz")) 
        
        # B) Decoy'lar (Non-Binder'lar)
        decoy_files = glob.glob(os.path.join(folder_path, "decoys_final.sdf.gz"))
        
        # Mol2 dosyalarının varlığını kontrol et
        if not active_files or not decoy_files:
            print(f"UYARI: {target_name} için aktif veya decoy sdf.gz dosyaları bulunamadı. Atlanıyor.")
            continue
            
        # 3. Adım: Haritaya kaydet
        
        # DUD-E'deki moleküllerin gerçek ID'si yoktur, bu yüzden "target_name/type" formatında key oluşturulur.
        # Bu dosyalar sıkıştırılmış olduğundan (mol2.gz), bu scriptin çalışmasından önce açılmaları gerekir 
        # veya dosya okuyucu modülün .gz formatını desteklemesi gerekir. 
        # Basitlik için, dosyanın .mol2 veya .sdf uzantılı (açılmış) olduğunu varsayalım.
        
        # Planetin bu sıkıştırılmış dosyaları açıp tek tek molekülleri okuyabilmesi gerekir.
        # Biz burada sadece dosya yollarını haritalıyoruz.

        # Aktifler için giriş:
        structure_map[f"{target_name}_actives"] = {
            "target_name": target_name,
            "mol_type": "active",
            "is_binder": 1.0, # Bu tarama gücünde True Label'dır
            "protein_pdb": protein_pdb,
            "ligand_sdfgz": active_files[0]
        }
        found_molecules += 1 # Sadece bu dosya binlerce molekül içerir!
        
        # Decoylar için giriş:
        structure_map[f"{target_name}_decoys"] = {
            "target_name": target_name,
            "mol_type": "decoy",
            "is_binder": 0.0, # Bu tarama gücünde False Label'dır
            "protein_pdb": protein_pdb,
            "ligand_sdfgz": decoy_files[0]
        }
        found_molecules += 1 # Sadece bu dosya binlerce molekül içerir!
        
    # 4. Adım: Haritayı JSON dosyasına kaydet
    print(f"Toplam {found_molecules} DUD-E hedef seti (aktif/decoy grupları) JSON dosyasına yazılıyor: {output_json}")
    with open(output_json, 'w') as f:
        json.dump(structure_map, f, indent=4)
        
    print("İşlem tamamlandı. DUD-E yapısal haritası oluşturuldu.")
    return structure_map

# --- Çalıştırma ---
if __name__ == '__main__':
    # BASE_DIR'ı kendi dosya yapınıza göre düzeltin.
    # Sizin için doğru yol muhtemelen: 
    # BASE_DIR = "/media/ubuntu/8TB/havvanur/PLANET_MODEL/data/datasets/all"
    
    # OUTPUT_JSON'ı da uygun bir yere güncelleyin.
    BASE_DIR_PATH = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/all"
    
    # JSON dosyasını DUD-E klasörünün bir üstüne kaydet
    OUTPUT_JSON_PATH = os.path.join(os.path.dirname(BASE_DIR_PATH), "DUD_E_structure_map_prepared.json")

    create_dude_structure_map(BASE_DIR_PATH, OUTPUT_JSON_PATH)

DUD-E hedefleri taranıyor (102 adet)...
UYARI: fgfr1 için aktif veya decoy sdf.gz dosyaları bulunamadı. Atlanıyor.
Toplam 202 DUD-E hedef seti (aktif/decoy grupları) JSON dosyasına yazılıyor: /arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/DUD_E_structure_map_prepared.json
İşlem tamamlandı. DUD-E yapısal haritası oluşturuldu.


In [ ]:
import os, argparse, pickle, random, json
import sys
from functools import partial
import multiprocessing 
from multiprocessing import Pool
from tqdm import tqdm
from typing import List, Tuple
import glob
import gzip # KRİTİK: gzip modülü burada kullanılacak!

# --- RDKit ve Kütüphane Yükleme ---
# ... (Import kısmı aynı kalır) ...
try:
    from rdkit import Chem
    from rdkit.Chem import AllChem
    from rdkit import RDLogger
    RDLogger.DisableLog('rdApp.info')
    RDLogger.DisableLog('rdApp.warning')
    # chemutils'den ComplexPocket ve Mol sınıflarınızın doğru yüklendiğini varsayıyoruz
    from chemutils import ComplexPocket, Mol 
except ImportError as e:
    # Bu hata büyük ihtimalle RDKit'ten kaynaklanmaktadır.
    print(f"KRİTİK HATA: Kütüphane bulunamadı. Detay: {e}", file=sys.stderr)
    sys.exit(1)

# --- SABİT YOLLAR VE AYARLAR ---
# DUD-E haritanızın yolu
DUD_E_MAP_PATH_LOCAL = '/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/DUD_E_structure_map_prepared.json'
DEFAULT_PK = 0.0 


def process_DUD_E_Target(record_key: str, DUD_E_data: dict, output_dir: str) -> str:
    # --- 1. Veri Çıkarımı ---
    try:
        record = DUD_E_data[record_key]
        target_name = record['target_name']
        mol_type = record['mol_type'] 
        is_binder = record['is_binder'] 
        protein_pdb = record['protein_pdb']
        ligand_path_gz = record['ligand_sdfgz'] 
    except KeyError as e:
        return f"UYARI: {record_key} JSON haritasında yok veya anahtar eksik: {e}."

    target_output_dir = os.path.join(output_dir, target_name)
    os.makedirs(target_output_dir, exist_ok=True)
    
    success_count = 0
    mol_count = 0
    supplier = None
    
    # -----------------------------------------------------------
    # 2. KRİTİK BÖLÜM: RDKIT SDMOLSUPPLIER DÜZELTMESİ
    # -----------------------------------------------------------
    sdf_content = None
    try:
        # A. GZIP dosyasını aç ve tüm içeriği belleğe oku (metin dizesi olarak)
        with gzip.open(ligand_path_gz, 'rt', encoding='utf-8') as gz_file:
            sdf_content = gz_file.read()
        
        # B. SDMolSupplier yerine, dize içeriğinden okuma işlevini kullanan MolFromDataStream() veya 
        # ilgili bir yapıyı kullanmalıyız. Ancak SetData kullanıyorsak:
        
        supplier = Chem.SDMolSupplier() 
        supplier.SetData(sdf_content, removeHs=False, sanitize=False, strictParsing=False) 
        
        # *** HATA VEREN KISMI KALDIRIYORUZ ***
        # if supplier is None or supplier.GetNumMols() == 0: 
        #      raise ValueError("SDMolSupplier SetData ile 0 molekül okudu.")
        
    except Exception as e:
        sys.stderr.write(f"MOL OKUYUCU HATA [{record_key}]: {ligand_path_gz} -> KRİTİK Okuma Hatası (SetData): {e}. Atlanıyor.\n")
        return f"❌ HATA: {record_key} MolSupplier oluşturulamadı/okunamadı."

    # -----------------------------------------------------------
    # 3. MOLEKÜL DÖNGÜSÜ VE PKL OLUŞTURMA
    # -----------------------------------------------------------
    
    try:
        for mol_rdkit in supplier:
            
            # 1. RDKit'in kendi atladığı bozuk/geçersiz molekülleri atla
            if mol_rdkit is None:
                mol_count += 1
                continue

            # 2. Her molekülü tek tek try/except ile sar! (ComplexPocket hatalarını yakalamak için)
            try:
                # PKL dosya yolu ve isimlendirme
                mol_name = f"{target_name}_{mol_type}_{mol_count:06d}"
                pkl_path = os.path.join(target_output_dir, f"{mol_name}.pkl")
                
                if os.path.exists(pkl_path):
                     success_count += 1
                     mol_count += 1
                     continue
                
                # --- ComplexPocket Oluşturma ---
                mol_obj = Mol(mol_rdkit) 
                pocket = ComplexPocket(protein_pdb, mol_obj, is_binder) 
                
                # Kaydet
                with open(pkl_path, 'wb') as f:
                    pickle.dump(pocket, f, pickle.HIGHEST_PROTOCOL)
                    
                success_count += 1
                
            except Exception as inner_e:
                sys.stderr.write(f"İŞLEME HATA [{record_key}, Mol #{mol_count}]: Hata: {inner_e}. Atlanıyor.\n")
                
            mol_count += 1 # Hata olsa bile sayacı artır

    except Exception as outer_e:
        sys.stderr.write(f"ANA DÖNGÜ HATASI [{record_key}]: {outer_e}\n")
        return f"❌ KRİTİK İŞLEME HATA: {record_key}. Detay: {outer_e}"


    return f"✅ İŞLENDİ: {record_key} ({mol_type}) - Toplam {success_count} molekül PKL olarak kaydedildi."
    
def collect_all_pkl_paths(map_keys: List[str], output_dir: str) -> List[Tuple[str, float]]:
    """Tüm yeni oluşturulan PKL dosyalarının yollarını ve is_binder etiketlerini toplar."""
    all_test_records = []
    
    # 1. Haritayı yükle
    try:
        with open(DUD_E_MAP_PATH_LOCAL, 'r') as f:
            DUD_E_data = json.load(f)
    except FileNotFoundError:
        print(f"KRİTİK HATA: DUD-E JSON dosyası bulunamadı: {DUD_E_MAP_PATH_LOCAL}", file=sys.stderr)
        return []

    print("PKL Yolları Toplanıyor...")
    
    for record_key in tqdm(map_keys):
        record = DUD_E_data.get(record_key, {})
        target_name = record.get('target_name')
        is_binder = record.get('is_binder', DEFAULT_PK) # is_binder etiketi (1.0 veya 0.0)
        
        if not target_name:
            continue
            
        target_output_dir = os.path.join(output_dir, target_name)
        
        # 2. Oluşturulan PKL'leri tara
        # Dosya adı formatı: {target_name}_{mol_type}_{mol_idx:06d}.pkl
        pkl_pattern = os.path.join(target_output_dir, f"{target_name}*_{int(is_binder)}_*pkl") 
        
        for pkl_path in glob.glob(pkl_pattern):
             # Tuple: (PKL Yolu, is_binder Etiketi)
             all_test_records.append((pkl_path, is_binder)) 
             
    return all_test_records

# process_DUD_E.py dosyasındaki main() fonksiyonu içinde yapılacak düzenlemeler:

def main():
    # Eski argparse bloğunu bu yeni sabit blokla DEĞİŞTİRİN:
    # ----------------------------------------------------------------
    # parser = argparse.ArgumentParser()
    # parser.add_argument('-o', '--output_path', default="DUD_E_records_for_testing.pkl", help="...")
    # parser.add_argument('-n', '--njobs', required=True, type=int, help="...")
    # parser.add_argument('-d', '--dud_e_dir', required=True, help="...")
    # args = parser.parse_args()
    # ----------------------------------------------------------------
    
    # 🌟 YENİ SABİT AYARLAR (Jupyter için) 🌟
    class Args:
        # LÜTFEN BU YOLLARI KENDİ YAPINIZA GÖRE GÜNCELLEYİN!
        # DUD-E klasörlerinin bir üstü, JSON'un bulunduğu yer.
        dud_e_dir = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets" 
        output_path = "DUD_E_test_records.pkl"
        njobs = 48 # Kullanmak istediğiniz CPU çekirdek sayısı
    args = Args()
    # ----------------------------------------------------------------

    # DUD-E map dosyasının yolu (Jupyter'de global değil, yerel değişken)
    global DUD_E_MAP_PATH_LOCAL
    DUD_E_MAP_PATH_LOCAL = os.path.join(args.dud_e_dir, "DUD_E_structure_map_prepared.json")
    
    # Her hedef (target) için PKL'lerin kaydedileceği ana dizin
    PKL_OUTPUT_ROOT = os.path.join(args.dud_e_dir, "DUD_E_PKL_Processed")
    os.makedirs(PKL_OUTPUT_ROOT, exist_ok=True)
    
    try:
        with open(DUD_E_MAP_PATH_LOCAL, 'r') as f:
            DUD_E_data = json.load(f)
    except FileNotFoundError:
        print(f"KRİTİK HATA: DUD-E JSON dosyası bulunamadı: {DUD_E_MAP_PATH_LOCAL}", file=sys.stderr)
        sys.exit(1)
        
    map_keys = list(DUD_E_data.keys())
    
    njobs = args.njobs
    print(f"Toplam {len(map_keys)} DUD-E hedef seti (aktif/decoy grupları) işleniyor ({njobs} çekirdek).")

    # --- AŞAMA 1: COMPLEX PKL OLUŞTURMA ---
    with Pool(njobs) as pool:
        complex_worker_func = partial(process_DUD_E_Target, DUD_E_data=DUD_E_data, output_dir=PKL_OUTPUT_ROOT)
        
        # tqdm progress barı ile paralel işleme
        _ = list(tqdm(pool.imap_unordered(complex_worker_func, map_keys), 
                      total=len(map_keys), 
                      desc="DUD-E PKL'ler Oluşturuluyor"))
    
    # --- AŞAMA 2: PKL YOLLARINI TOPLAMA ---
    print("\n--- AŞAMA 2: Oluşturulan tüm PKL yolları toplanıyor ---")
    all_test_records = collect_all_pkl_paths(map_keys, PKL_OUTPUT_ROOT)
    
    # Sonuç listesini kaydet
    final_output_path = os.path.join(args.dud_e_dir, args.output_path)
    with open(final_output_path, 'wb') as out_f:
        pickle.dump(all_test_records, out_f, pickle.HIGHEST_PROTOCOL)
        
    print(f"\n✅ BAŞARILI: Toplam {len(all_test_records)} molekül kaydı ({final_output_path} dosyasına) kaydedildi.")
    print(f"Bu dosya, PLANET'in ProLigDataset'ine girdi olarak verilebilir.")


if __name__ == '__main__':
    # Multiprocessing başlangıç metodu ayarı
    if os.name == 'posix':
        try:
            # 'fork' metodu mevcut değilse veya kararsızsa 'spawn' kullan
            if 'fork' not in multiprocessing.get_all_start_methods():
                multiprocessing.set_start_method('spawn', force=True)
        except RuntimeError:
            pass
            
    main()

Toplam 202 DUD-E hedef seti (aktif/decoy grupları) işleniyor (48 çekirdek).


DUD-E PKL'ler Oluşturuluyor:   0%|          | 0/202 [00:00<?, ?it/s]MOL OKUYUCU HATA [ada_actives]: /arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/all/ada/actives_final.sdf.gz -> KRİTİK Okuma Hatası (SetData): 'SDMolSupplier' object has no attribute 'GetNumMols'. Atlanıyor.
MOL OKUYUCU HATA [kpcb_actives]: /arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/all/kpcb/actives_final.sdf.gz -> KRİTİK Okuma Hatası (SetData): 'SDMolSupplier' object has no attribute 'GetNumMols'. Atlanıyor.
MOL OKUYUCU HATA [aofb_actives]: /arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/all/aofb/actives_final.sdf.gz -> KRİTİK Okuma Hatası (SetData): 'SDMolSupplier' object has no attribute 'GetNumMols'. Atlanıyor.
MOL OKUYUCU HATA [tryb1_actives]: /arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/all/tryb1/actives_final.sdf.gz -> KRİTİK Okuma Hatası (SetData): 'SDMolSupplier' object has no attribute 'GetNumMols'. Atlanıyor.
MOL OKUYUCU HATA [gria2_actives]: /arf/scratch/h


--- AŞAMA 2: Oluşturulan tüm PKL yolları toplanıyor ---
PKL Yolları Toplanıyor...


100%|██████████| 202/202 [00:00<00:00, 2280.22it/s]


✅ BAŞARILI: Toplam 0 molekül kaydı (/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/DUD_E_test_records.pkl dosyasına) kaydedildi.
Bu dosya, PLANET'in ProLigDataset'ine girdi olarak verilebilir.


In [1]:
import os
import gzip
import tempfile
from rdkit import Chem
from chemutils import Mol # Sizin Mol sınıfınızın burada olduğunu varsayıyoruz.

# --- KRİTİK YOLLAR ---
LIGAND_PATH_GZ = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/all/bace1/actives_final.sdf.gz"
PROTEIN_PATH = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/all/bace1/receptor.pdb"

# --- 1. GZIP Dosyasını Diske Açma ---
temp_dir = "/tmp/dude_test" # Geçici klasör oluşturun
os.makedirs(temp_dir, exist_ok=True)
TEMP_SDF_PATH = os.path.join(temp_dir, "bace1_actives_unzipped.sdf")

print("1. GZIP dosyası diske açılıyor...")
try:
    with gzip.open(LIGAND_PATH_GZ, 'rb') as f_in:
        sdf_content = f_in.read()
    with open(TEMP_SDF_PATH, 'wb') as f_out:
        f_out.write(sdf_content)
    print(f"✅ Başarılı: Dosya diske açıldı: {TEMP_SDF_PATH}")
except Exception as e:
    print(f"❌ HATA: Dosya açılırken/yazılırken hata oluştu: {e}")
    sys.exit(1)

# --- 2. RDKit ile Açılmış SDF Dosyasını Oku (SADECE DOSYA YOLU İLE) ---
valid_mol_count = 0
error_count = 0

print("\n2. Açılmış SDF dosyası RDKit ile okunuyor...")
try:
    # Supplier'ı doğrudan DISK ÜZERİNDEKİ SDF dosyasıyla başlatıyoruz
    supplier = Chem.SDMolSupplier(
        TEMP_SDF_PATH, 
        removeHs=False, 
        sanitize=False, 
        strictParsing=False
    )
    
    # Mol döngüsünü başlat
    for mol_rdkit in supplier:
        if mol_rdkit is None:
            error_count += 1
            continue
            
        try:
            # Sizin Mol sınıfınızın testini yap
            mol_obj = Mol(mol_rdkit) 
            
            # ComplexPocket yaratmayı dene (Çalışıp çalışmadığını görmek için)
            # DUD-E'de is_binder = 1.0 (Aktif)
            pocket = ComplexPocket(PROTEIN_PATH, mol_obj, pK=1.0)
            
            valid_mol_count += 1
        except Exception:
            # Sizin Mol() veya ComplexPocket() sınıfınızda hata varsa buraya düşer
            error_count += 1

except Exception as e:
    print(f"❌ KRİTİK OKUMA HATASI: SDMolSupplier başlatılırken hata oluştu: {e}")

finally:
    # Geçici dosyayı sil
    if os.path.exists(TEMP_SDF_PATH):
        os.remove(TEMP_SDF_PATH)

print(f"\n--- SONUÇ ---")
print(f"Başarılı Okunan Molekül Sayısı (Diskten): {valid_mol_count}")
print(f"Atlanan (Hatalı) Molekül Sayısı: {error_count}")

1. GZIP dosyası diske açılıyor...
✅ Başarılı: Dosya diske açıldı: /tmp/dude_test/bace1_actives_unzipped.sdf

2. Açılmış SDF dosyası RDKit ile okunuyor...

--- SONUÇ ---
Başarılı Okunan Molekül Sayısı (Diskten): 0
Atlanan (Hatalı) Molekül Sayısı: 485


In [2]:
import os
from rdkit import Chem

# LÜTFEN HATA VEREN DOSYANIN TAM YOLUNU KULLANIN
LIGAND_PATH = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/all/akt1/crystal_ligand.mol2"

# --- 1. Sadece MolSupplier ile Okuma ---
print(f"1. {LIGAND_PATH} MolSupplier ile okunuyor...")

try:
    # MolSupplier normalde .mol2 ve .sdf dosyalarını okumak için tasarlanmıştır.
    supplier = Chem.SDMolSupplier(
        LIGAND_PATH, 
        removeHs=False, 
        sanitize=False, 
        strictParsing=False
    )
    
    mol_list = [mol for mol in supplier if mol is not None]

    if not mol_list:
        print("❌ HATA: MolSupplier dosyayı okudu ancak geçerli molekül bulamadı (Hepsi None).")
        # Dosya bozuktur veya formatı SDMolSupplier için uygun değildir.
    else:
        print(f"✅ BAŞARILI: MolSupplier {len(mol_list)} adet geçerli molekül okudu.")
        
except Exception as e:
    print(f"❌ KRİTİK HATA: MolSupplier başlatılamadı: {e}")

# --- 2. Alternatif Okuma Yöntemi (MolFromMolFile) ---
# .mol2 için bazen daha kararlı olabilir.
print(f"\n2. {LIGAND_PATH} MolFromMolFile ile okunuyor...")
try:
    mol_from_file = Chem.MolFromMolFile(LIGAND_PATH, sanitize=False, removeHs=False)
    
    if mol_from_file is None:
        print("❌ HATA: MolFromMolFile okuma başarısız oldu (None döndü).")
    else:
        print(f"✅ BAŞARILI: MolFromMolFile molekülü okudu. Atom sayısı: {mol_from_file.GetNumAtoms()}")

except Exception as e:
    print(f"❌ KRİTİK HATA: MolFromMolFile çağrısı çöktü: {e}")

1. /arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/all/akt1/crystal_ligand.mol2 MolSupplier ile okunuyor...
❌ HATA: MolSupplier dosyayı okudu ancak geçerli molekül bulamadı (Hepsi None).

2. /arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/all/akt1/crystal_ligand.mol2 MolFromMolFile ile okunuyor...
❌ HATA: MolFromMolFile okuma başarısız oldu (None döndü).


[01:11:49] ERROR: Counts line too short: 'SMALL' on line4
[01:11:49] ERROR: moving to the beginning of the next molecule
[01:11:49] Counts line too short: 'SMALL' on line4


In [6]:
import os
import sys
import numpy as np
import torch
from rdkit import Chem
from rdkit.Chem import AllChem 

# Lütfen chemutils.py dosyasındaki ComplexPocket, Mol, ProteinPocket ve diğer yardımcı
# sınıflarınızın ve fonksiyonlarınızın (near_pocket, mass_center_from_pdb, Residue)
# bu ortamda erişilebilir olduğundan emin olun (ya import edin ya da buraya kopyalayın).
# Hata vermemesi için chemutils'i import ediyoruz:
try:
    from chemutils import ProteinPocket, Mol, ComplexPocket 
except ImportError:
    print("HATA: chemutils import edilemedi. Lütfen tüm sınıfları (Mol, ProteinPocket, vb.) bu ortama kopyalayın.")
    sys.exit(1)


# --- HEDEFE ÖZEL PATH'LER ---
TARGET_NAME = "akt1"
DUDE_DATA_ROOT = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/all"
TARGET_DIR = os.path.join(DUDE_DATA_ROOT, TARGET_NAME)

PROTEIN_PDB = os.path.join(TARGET_DIR, "receptor.pdb")
LIGAND_MOL2 = os.path.join(TARGET_DIR, "crystal_ligand.mol2")

print(f"--- {TARGET_NAME} Hedefi Test Ediliyor ---")

try:
    # 1. ProteinPocket Objesi Oluşturma Denemesi (pocket merkezini bulur)
    # Bu adım, daha önce ValueError fırlatmıştı.
    print(f"1. ProteinPocket'in {LIGAND_MOL2} ile başlatılması deneniyor...")
    
    pocket = ProteinPocket(
        protein_pdb=PROTEIN_PDB,
        ligand_sdf=LIGAND_MOL2 # Artık MolFromMol2File kullanmalı
    )
    
    print("✅ ADIM 1 BAŞARILI: ProteinPocket objesi oluşturuldu.")
    print(f"   Bulunan Cep Residü Sayısı: {pocket.res_count}")
    
    # 2. Ligand Okuma ve Merkez Koordinat Kontrolü
    # Ligand, pocket objesinin içinde Mol objesi olarak saklanmış olmalı.
    center = pocket.ligand.compute_centeroid()
    print(f"✅ ADIM 2 BAŞARILI: Ligand Merkezi (Centeroid) Hesaplandı: {center}")
    
    
    # 3. Temel Mol Fonksiyonu Kontrolü (Daha önce AttributeError veren GetNumAtoms)
    atom_count = pocket.ligand.mol.GetNumAtoms()
    print(f"✅ ADIM 3 BAŞARILI: Ligand Atom Sayısı: {atom_count}")


    print("\n--- TEST SONUCU: BAŞARILI ---")
    print("ProteinPocket artık .mol2 dosyasını işleyebiliyor.")

except ValueError as e:
    # Bozuk veri nedeniyle atlatma (Bu, beklenen davranış olabilir)
    print(f"\n❌ TEST SONUCU: BAŞARISIZ (ValueError)")
    print(f"Kritik Hata: {e}")
    print("Bu, .mol2 dosyasının RDKit'in MolFromMol2File fonksiyonu için bile bozuk olduğu anlamına gelir.")
    
except Exception as e:
    print(f"\n❌ TEST SONUCU: KRİTİK PYTHON HATASI")
    print(f"Beklenmeyen Hata: {type(e).__name__}: {e}")

--- akt1 Hedefi Test Ediliyor ---
1. ProteinPocket'in /arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/all/akt1/crystal_ligand.mol2 ile başlatılması deneniyor...

❌ TEST SONUCU: BAŞARISIZ (ValueError)
Kritik Hata: Input object is not a valid RDKit Mol (GetNumAtoms missing).
Bu, .mol2 dosyasının RDKit'in MolFromMol2File fonksiyonu için bile bozuk olduğu anlamına gelir.


[01:19:47] ERROR: Cannot convert 'USE' to unsigned int on line 4
[01:19:47] ERROR: moving to the beginning of the next molecule


In [7]:
import os
import subprocess
from rdkit import Chem

LIGAND_PATH = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/all/akt1/crystal_ligand.mol2"

print(f"--- 1. Dosya Düzeltme İşlemi ({LIGAND_PATH}) ---")

# sed komutunu Python subprocess modülü ile çalıştırıyoruz.
# '/@<TRIPOS>MOLECULE/ {n;n;n;d}' -> '@<TRIPOS>MOLECULE' satırını bul, sonraki üç satırı atla (n;n;n), dördüncü satırı sil (d).
try:
    # Komut: @<TRIPOS>MOLECULE satırından sonra gelen dördüncü satırı sil.
    # Bu, hatalı 'SMALL' satırını hedef almalıdır.
    subprocess.run(
        ['sed', '-i', '/@<TRIPOS>MOLECULE/ {n;n;n;d}', LIGAND_PATH], 
        check=True, 
        capture_output=True, 
        text=True
    )
    print("✅ Düzeltme Başarılı: Hatalı 'SMALL' satırı silinmiş olmalıdır.")

except subprocess.CalledProcessError as e:
    print(f"❌ HATA: sed komutu başarısız oldu: {e.stderr}")
    sys.exit(1)

# --- 2. Düzeltilmiş Dosyayı Tekrar Okuma ---
print("\n--- 2. Düzeltilmiş Dosya RDKit Testi ---")

try:
    # Dosya MolSupplier yerine MolFromMol2File ile okunmalıdır.
    mol_from_file = Chem.MolFromMol2File(LIGAND_PATH, sanitize=False, removeHs=False)
    
    if mol_from_file is None:
        print("❌ TEST SONUCU: BAŞARISIZ. Düzeltme sonrası bile RDKit None döndürdü.")
    else:
        print(f"✅ TEST SONUCU: BAŞARILI! Molekül okundu. Atom sayısı: {mol_from_file.GetNumAtoms()}")

except Exception as e:
    print(f"❌ KRİTİK PYTHON HATASI: {type(e).__name__}: {e}")

--- 1. Dosya Düzeltme İşlemi (/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/all/akt1/crystal_ligand.mol2) ---
✅ Düzeltme Başarılı: Hatalı 'SMALL' satırı silinmiş olmalıdır.

--- 2. Düzeltilmiş Dosya RDKit Testi ---
✅ TEST SONUCU: BAŞARILI! Molekül okundu. Atom sayısı: 30


In [15]:
import os
import numpy as np
from rdkit import Chem
from chemutils import ProteinPocket # Sizin sınıflarınızı import eder

LIGAND_MOL2 = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/all/akt1/crystal_ligand.mol2"
PROTEIN_PDB = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/all/akt1/receptor.pdb"
DISTANCE_CUTOFF = 12.0

# Ligand merkezini hesapla
mol = Chem.MolFromMol2File(LIGAND_MOL2, sanitize=False, removeHs=False)
# Düzeltme: centeriod_x, y, z'yi None bırakın ve ligand_sdf'i doğru path ile verin.
pocket = ProteinPocket(
    protein_pdb=PROTEIN_PDB, 
    ligand_sdf=LIGAND_MOL2 
) 
centeroid = pocket.ligand.compute_centeroid()
print(f"Ligand Merkezi (Centeroid): {centeroid}")

protein_coordinates = []
with open(PROTEIN_PDB, 'r') as f:
    for line in f:
        if line.startswith('ATOM'):
            try:
                # ProteinPocket/near_pocket'inizdeki dizinlemeyi kullanın
                x = float(line[30:38].strip()) 
                y = float(line[38:46].strip())
                z = float(line[46:54].strip())
                protein_coordinates.append(np.array([x, y, z], dtype=np.float32))
            except ValueError:
                # PDB okuma hatası varsa atla
                continue

protein_coordinates = np.array(protein_coordinates)
if len(protein_coordinates) == 0:
    print("❌ HATA: PDB dosyasında geçerli ATOM koordinatı bulunamadı.")
    sys.exit(1)

# Minimum mesafeyi hesapla
distances = np.sqrt(np.sum(np.square(protein_coordinates - centeroid), axis=1))
min_distance = np.min(distances)
near_residues_count = np.sum(distances <= DISTANCE_CUTOFF)

print(f"Toplam Protein Atom Sayısı: {len(protein_coordinates)}")
print(f"Merkeze En Yakın Mesafe: {min_distance:.3f} Å")
print(f"Cep (Pocket) İçindeki Atom Sayısı ({DISTANCE_CUTOFF} Å): {near_residues_count}")

if min_distance > 15.0 or near_residues_count == 0:
    print("⚠️ UYARI: Cep Atom Sayısı Sıfır. ProteinPocket'in boş kalması bu yüzden.")

[01:40:12] ERROR: CTAB version string invalid at line 4
[01:40:12] ERROR: moving to the beginning of the next molecule


ValueError: Input object is not a valid RDKit Mol (GetNumAtoms missing).

In [23]:
# Lütfen bu kod bloğunu Python yorumlayıcısında çalıştırın
import openbabel # Doğru paketi Conda kurduysa bu çalışmalıdır
import os
from rdkit import Chem # Dönüşüm sonrası kontrol için

# Dosya yolları
LIGAND_MOL2 = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/all/akt1/crystal_ligand.mol2"
LIGAND_SDF = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/all/akt1/crystal_ligand.sdf"

try:
    # Conda ile gelen OpenBabel paketi (genellikle 'openbabel' modülünü kullanır)
    # Bu, komut satırı aracının yaptığı aynı işlemi Python koduyla yapar.
    
    # Yeni yaklaşım: openbabel'in içindeki pybel'i kullanmayı deniyoruz (bazen bu şekilde erişilir)
    from openbabel import pybel
    
    mols = list(pybel.readfile("mol2", LIGAND_MOL2))
    
    if not mols:
        raise ValueError("Pybel/OpenBabel molekül okuyamadı.")

    mol = mols[0]
    mol.write("sdf", LIGAND_SDF, overwrite=True)
    print("✅ Pybel Dönüşümü Başarılı.")
    
    # RDKit ile kontrol:
    mol_rdkit = Chem.SDMolSupplier(LIGAND_SDF, sanitize=False)[0]
    print(f"✅ RDKit Kontrolü Başarılı. Atom Sayısı: {mol_rdkit.GetNumAtoms()}")

except Exception as e:
    print(f"❌ HATA: Nihai Dönüşüm Başarısız: {e}")

✅ Pybel Dönüşümü Başarılı.
✅ RDKit Kontrolü Başarılı. Atom Sayısı: 30


In [24]:
import os
import sys
from openbabel import pybel # Conda ile kurduğunuz paket
from rdkit import Chem # Kontrol için

DUDE_ROOT = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/all"
SUCCESS_COUNT = 0
FAIL_COUNT = 0

print(f"--- {DUDE_ROOT} Altındaki Hedefler Taranıyor ---")

for target_name in os.listdir(DUDE_ROOT):
    target_path = os.path.join(DUDE_ROOT, target_name)
    
    # Sadece dizinleri işle
    if not os.path.isdir(target_path):
        continue
    
    mol2_path = os.path.join(target_path, 'crystal_ligand.mol2')
    sdf_path = os.path.join(target_path, 'crystal_ligand.sdf')
    
    if not os.path.exists(mol2_path):
        # crystal_ligand.mol2 yoksa atla
        continue
    
    print(f"\n[İŞLENİYOR]: {target_name}")

    try:
        # 1. Pybel ile okuma
        mols = list(pybel.readfile("mol2", mol2_path))
        
        if not mols:
            raise ValueError("Pybel dosyadan molekül okuyamadı (None döndü).")

        # 2. SDF olarak kaydetme
        mol = mols[0]
        mol.write("sdf", sdf_path, overwrite=True)
        
        # 3. RDKit ile kontrol (Garanti için)
        mol_rdkit = Chem.SDMolSupplier(sdf_path, sanitize=False)[0]
        if mol_rdkit is None or mol_rdkit.GetNumAtoms() == 0:
            raise ValueError("RDKit, oluşturulan SDF'i okuyamadı.")

        print(f"✅ BAŞARILI: {target_name} -> Atom Sayısı: {mol_rdkit.GetNumAtoms()}")
        SUCCESS_COUNT += 1

    except Exception as e:
        print(f"❌ ATLANDI: {target_name} ({mol2_path}): {type(e).__name__}: {e}")
        FAIL_COUNT += 1

print(f"\n--- İŞLEM SONUCU ---")
print(f"Başarılı Dönüşüm: {SUCCESS_COUNT} hedef")
print(f"Bozuk Veri (Atlandı): {FAIL_COUNT} hedef")

--- /arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/all Altındaki Hedefler Taranıyor ---

[İŞLENİYOR]: kpcb
✅ BAŞARILI: kpcb -> Atom Sayısı: 62

[İŞLENİYOR]: vgfr2
✅ BAŞARILI: vgfr2 -> Atom Sayısı: 50

[İŞLENİYOR]: ada
✅ BAŞARILI: ada -> Atom Sayısı: 45

[İŞLENİYOR]: dpp4
✅ BAŞARILI: dpp4 -> Atom Sayısı: 49

[İŞLENİYOR]: tryb1
✅ BAŞARILI: tryb1 -> Atom Sayısı: 51

[İŞLENİYOR]: ppara
✅ BAŞARILI: ppara -> Atom Sayısı: 53

[İŞLENİYOR]: glcm
✅ BAŞARILI: glcm -> Atom Sayısı: 34

[İŞLENİYOR]: aofb
✅ BAŞARILI: aofb -> Atom Sayısı: 32

[İŞLENİYOR]: gria2
✅ BAŞARILI: gria2 -> Atom Sayısı: 40

[İŞLENİYOR]: pde5a
✅ BAŞARILI: pde5a -> Atom Sayısı: 33

[İŞLENİYOR]: pygm
✅ BAŞARILI: pygm -> Atom Sayısı: 49

[İŞLENİYOR]: hivpr
✅ BAŞARILI: hivpr -> Atom Sayısı: 85

[İŞLENİYOR]: ppard
✅ BAŞARILI: ppard -> Atom Sayısı: 59

[İŞLENİYOR]: cp3a4
✅ BAŞARILI: cp3a4 -> Atom Sayısı: 96

[İŞLENİYOR]: hdac8
✅ BAŞARILI: hdac8 -> Atom Sayısı: 39

[İŞLENİYOR]: egfr
✅ BAŞARILI: egfr -> Atom Sayısı: 60

[İŞ

In [31]:
import os
import sys
import numpy as np
from rdkit import Chem
# chemutils'den gerekli sınıfları import edin
from chemutils import ProteinPocket, Mol

# --- HEDEFE ÖZEL PATH'LER ---
TARGET_NAME = "akt1"
DUDE_DATA_ROOT = "/arf/scratch/hnaydin/PLANET_ORIGINAL/Planet_model/datasets/all"
TARGET_DIR = os.path.join(DUDE_DATA_ROOT, TARGET_NAME)

PROTEIN_PDB = os.path.join(TARGET_DIR, "receptor.pdb")
# Artık .sdf dosyasını kullanıyoruz (çünkü .mol2 bozuktu)
LIGAND_FILE = os.path.join(TARGET_DIR, "crystal_ligand.sdf") 
CUTOFF_DISTANCE = 12.0 # Pocket belirleme mesafesi

print(f"--- 1. {TARGET_NAME} Hedefi İçin ProteinPocket Başlatılıyor ---")

try:
    # ProteinPocket'ı başlatma (Ligand okuma ve centeroid hesaplama burada gerçekleşir)
    pocket = ProteinPocket(
        protein_pdb=PROTEIN_PDB,
        ligand_sdf=LIGAND_FILE
    )
    
    print("✅ ProteinPocket Başlatıldı.")

    # --- 2. Ligand Merkezini Kontrol Etme ---
    centeroid = pocket.ligand.compute_centeroid()
    print(f"\n✅ Ligand Merkezi (Centeroid) Koordinatları: {centeroid}")

    # --- 3. Cep Kalıntılarını Kontrol Etme ---
    # Bu, ProteinPocket'ın determine_pocket_residues metodunun sonucudur.
    res_count = pocket.res_count
    
    if res_count > 0:
        print(f"\n✅ BAŞARILI: {CUTOFF_DISTANCE} Å içinde {res_count} adet cep kalıntısı bulundu.")
        
        # İlk 5 kalıntıyı listele (örnek için)
        print("   Bulunan İlk 5 Kalıntı (Residue ID):")
        for i in range(min(5, res_count)):
            res = pocket.pocket_residues[i]
            # Residue objesinin __repr__ veya ilgili metotlarını kullanarak bilgi al
            print(f"   - {res.chainID}: {res.resName} {res.resID} (C-alpha: {res.alpha_position})")
            
    else:
        print(f"\n❌ HATA: {CUTOFF_DISTANCE} Å içinde hiçbir cep kalıntısı bulunamadı.")
        
    # --- 4. Protein Özelliklerini Kontrol Etme ---
    # np.concatenate hatası burada oluşuyordu, şimdi kontrol edelim.
    print(f"\n✅ Özellikler Boyutu Kontrolü (np.concatenate sonrası):")
    print(f"   Residue Özellik Matrisi Boyutu: {pocket.res_features.shape}")
    print(f"   C-alpha Koordinat Matrisi Boyutu: {pocket.alpha_coordinates.shape}")

except ValueError as e:
    print(f"\n❌ KRİTİK BAŞLATMA HATASI: {e}")
    print("Bu, Ligandın okunamadığı veya Ligand merkezi bulunsa bile proteinin bu merkez etrafında kalıntı içermediği anlamına gelir.")

except Exception as e:
    print(f"\n❌ BEKLENMEYEN HATA: {type(e).__name__}: {e}")

--- 1. akt1 Hedefi İçin ProteinPocket Başlatılıyor ---

❌ KRİTİK BAŞLATMA HATASI: need at least one array to concatenate
Bu, Ligandın okunamadığı veya Ligand merkezi bulunsa bile proteinin bu merkez etrafında kalıntı içermediği anlamına gelir.
